In [22]:
from openai import OpenAI, pydantic_function_tool
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

In [2]:
MODEL_ID="llama3.1:8b"

client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1",
)

The class name becomes the tool name.
The docstring and description tell the model when and how to use it. 

In [3]:
class GetWeather(BaseModel):
    """Get the current weather for one city."""
    city: str = Field(description="City name, e.g. Bishkek")

In [14]:
def get_weather(args: GetWeather) -> str:
    fake = {
        "Bishkek": "25C, shiny", 
        "London": "18C, rainy", 
        "Chicago": "22C, windy",
        "New York": "30C, cloudy",
        "Naryn": "10C, snowy"
    }
    return fake.get(args.city, "No data for this city.")

# city = GetWeather(city="Naryn")
# print(get_weather(city))

```pydantic_function_tool``` turns your class into the JSON schema the API expects

In [25]:
tools = [pydantic_function_tool(GetWeather)]
messages = [{
    "role": "user", "content": "Weather in Naryn and Tokmok?"
}]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    reasoning_effort="none",
    tools=tools,
)
assistant = response.choices[0].message

Step 4: Save the assistant message. The API needs to see what it asked for.

In [26]:
messages.append(assistant)

### Step 5: Answer every tool call.

In [27]:
for tool_call in assistant.tool_calls or []:
    args = GetWeather.model_validate_json(tool_call.function.arguments)
    result = get_weather(args)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result,
    })

* ```or []``` prevents a crash when there are no tool calls.
* ```model_validate_json``` converts the string into a checked Python object.
* ```tool_call_id``` matches each result to its request.

### Step 6: Get the final answer

In [28]:
final = client.chat.completions.create(
    model=MODEL_ID,
    reasoning_effort="none",
    messages=messages,
    tools=tools
)
final_output = final.choices[0].message.content
print(display(Markdown(final_output)))

You can get the current weather for Naryn and Tokmok. Naryn currently has a temperature of 10C and it is snowy. Unfortunately, I do not have any data for Tokmok.

None
